# N001 – LendingClub Dataset Discovery Summary

## Discovery Objective

The purpose of this notebook is to understand the LendingClub dataset before building:

* Clean Layer
* Core Warehouse
* Data Marts
* Feature Store
* Modeling Dataset
* Credit Risk Models

The objective is to eliminate assumptions and design the platform using evidence from the data.

---

# Dataset Overview

| Metric                   | Value                       |
| ------------------------ | --------------------------- |
| Dataset                  | LendingClub Accepted Loans  |
| Source File              | accepted_2007_to_2018Q4.csv |
| Raw Rows Loaded          | 2,260,701                   |
| Columns                  | 151                         |
| Valid Loan Records       | 2,260,668                   |
| Footer / Summary Records | 33                          |
| Primary Key              | id                          |
| member_id Population     | 0                           |
| Earliest Issue Date      | Apr-2008                    |
| Latest Issue Date        | Sep-2018                    |
| Application Types        | Individual, Joint App       |

---

# Dataset Grain

## Grain Definition

```text
1 Row = 1 Loan
```

Evidence:

| Metric            |     Value |
| ----------------- | --------: |
| Valid Loans       | 2,260,668 |
| Distinct Loan IDs | 2,260,668 |

Result:

```text
Loan ID is unique.
```

---

# Primary Key Assessment

## Candidate Key Analysis

| Column    | Distinct Values | Conclusion    |
| --------- | --------------: | ------------- |
| id        |       2,260,668 | Primary Key   |
| member_id |               0 | Invalid       |
| url       |       2,260,668 | Nearly Unique |
| loan_amnt |           1,572 | Not Unique    |

## Selected Primary Key

```text
id
```

---

# Deprecated Fields

## member_id

Findings:

| Metric            |     Value |
| ----------------- | --------: |
| Total Records     | 2,260,701 |
| Populated Records |         0 |

Decision:

```text
member_id is deprecated.
```

This field will not be used in:

* Cleaning
* Warehouse
* Marts
* Features
* Modeling

---

# Footer Records Discovery

The raw file contains 33 non-loan records.

Examples:

```text
Total amount funded in policy code 1: ...
Total amount funded in policy code 2: ...
Loans that do not meet the credit policy
```

These records are dataset summaries and not actual loans.

## Cleaning Rule #1

```sql
try_cast(id as bigint) is not null
```

Valid loan records satisfy this condition.

---

# Time Coverage

## Issue Date Coverage

| Metric             | Value    |
| ------------------ | -------- |
| Minimum Issue Date | Apr-2008 |
| Maximum Issue Date | Sep-2018 |

Observation:

Although the file name references 2007–2018Q4, valid loan records begin in Apr-2008.

---

# Application Types

| Application Type |     Loans |
| ---------------- | --------: |
| Individual       | 2,139,958 |
| Joint App        |   120,710 |

## Distribution

| Application Type | Percentage |
| ---------------- | ---------: |
| Individual       |     94.66% |
| Joint App        |      5.34% |

Decision:

```text
Joint application fields will be retained.
```

The population is sufficiently large to justify dedicated analysis.

---

# Business Entities and Attributes

## Entity 1 — Loan

Primary business entity.

### Attributes

```text
id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
issue_d
loan_status
purpose
title
policy_code
application_type
disbursement_method
initial_list_status
pymnt_plan
url
```

---

## Entity 2 — Borrower

Borrower demographic and financial profile.

### Attributes

```text
annual_inc
home_ownership
emp_title
emp_length
verification_status
dti
zip_code
addr_state
```

---

## Entity 3 — Credit Profile

Borrower credit bureau profile at origination.

### Attributes

```text
fico_range_low
fico_range_high
earliest_cr_line

delinq_2yrs
inq_last_6mths
open_acc
pub_rec
revol_bal
revol_util
total_acc

collections_12_mths_ex_med
mths_since_last_major_derog
acc_now_delinq
tot_coll_amt
tot_cur_bal

acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util

chargeoff_within_12_mths
delinq_amnt

mo_sin_old_il_acct
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl

mort_acc

mths_since_recent_bc
mths_since_recent_bc_dlq
mths_since_recent_inq
mths_since_recent_revol_delinq

num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_sats
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_sats

num_tl_120dpd_2m
num_tl_30dpd
num_tl_90g_dpd_24m
num_tl_op_past_12m

pct_tl_nvr_dlq
percent_bc_gt_75

pub_rec_bankruptcies
tax_liens

tot_hi_cred_lim
total_bal_ex_mort
total_bc_limit
total_il_high_credit_limit
total_rev_hi_lim
```

---

## Entity 4 — Payment Performance

Loan repayment and collections performance.

### Attributes

```text
out_prncp
out_prncp_inv

total_pymnt
total_pymnt_inv

total_rec_prncp
total_rec_int
total_rec_late_fee

recoveries
collection_recovery_fee

last_pymnt_d
last_pymnt_amnt

next_pymnt_d

last_credit_pull_d

last_fico_range_low
last_fico_range_high
```

---

## Entity 5 — Joint Applicant

Joint borrower information.

### Attributes

```text
annual_inc_joint
dti_joint
verification_status_joint

revol_bal_joint

sec_app_fico_range_low
sec_app_fico_range_high

sec_app_earliest_cr_line

sec_app_inq_last_6mths

sec_app_mort_acc
sec_app_open_acc
sec_app_revol_util
sec_app_open_act_il

sec_app_num_rev_accts

sec_app_chargeoff_within_12_mths
sec_app_collections_12_mths_ex_med

sec_app_mths_since_last_major_derog
```

---

## Entity 6 — Hardship Program

Borrowers enrolled in hardship assistance programs.

### Attributes

```text
hardship_flag
hardship_type
hardship_reason
hardship_status

hardship_amount
hardship_length
hardship_dpd

hardship_start_date
hardship_end_date

payment_plan_start_date

hardship_loan_status

deferral_term

orig_projected_additional_accrued_interest

hardship_payoff_balance_amount

hardship_last_payment_amount
```


## Entity 7 — Debt Settlement Program

Loans entering settlement.

### Attributes

```text
debt_settlement_flag
debt_settlement_flag_date

settlement_status
settlement_date

settlement_amount
settlement_percentage
settlement_term
```

---

## Entity 8 — Revolving / Installment Utilization

Advanced utilization and exposure metrics.

### Attributes

```text
open_acc_6m
open_act_il

open_il_12m
open_il_24m

mths_since_rcnt_il

total_bal_il
il_util

open_rv_12m
open_rv_24m

max_bal_bc

all_util

inq_fi
total_cu_tl
inq_last_12m
```

---

# Preliminary Entity Relationship Model

```text
Borrower
    |
    | 1:N
    |
Loan (Primary Entity)
    |
    +---- Credit Profile
    |
    +---- Payment Performance
    |
    +---- Joint Applicant
    |
    +---- Hardship Program
    |
    +---- Debt Settlement Program
    |
    +---- Utilization Metrics
```

---

# Major Data Quality Findings

## Finding 1

```text
member_id = 100% null
```

Impact:

Borrower-level identity cannot be constructed from source data.

---

## Finding 2

Joint applicant fields are sparse but important.

Approximately 5% of loans are joint applications.

---

## Finding 3

Hardship fields are highly sparse.

Only hardship participants populate these attributes.

---

## Finding 4

Settlement fields are highly sparse.

Only settlement cases populate these attributes.

---

## Finding 5

Footer records are embedded in the source file.

These must be removed during cleaning.

---

# Architectural Implications

## Important Discovery

The dataset does not contain a reliable borrower identifier.

Because:

```text
member_id = NULL for all records
```

the platform should initially be designed as:

```text
Loan-Centric
```

rather than:

```text
Borrower-Centric
```

until a stable borrower key can be derived.

---

# Discovery Decisions

| Decision          | Outcome            |
| ----------------- | ------------------ |
| Grain             | 1 Row = 1 Loan     |
| Primary Key       | id                 |
| member_id         | Deprecated         |
| Footer Records    | Remove             |
| Joint App Fields  | Retain             |
| Hardship Fields   | Retain             |
| Settlement Fields | Retain             |
| Warehouse Design  | Loan-Centric       |
| Next Phase        | Clean Layer Design |

---



```
```


In [2]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [3]:
ROOT = Path(
    r"D:\Project_Lighthouse"
)

DB_PATH = (
    ROOT
    / "projects"
    / "P0_Data_Platform"
    / "datasets"
    / "lendingclub"
    / "data"
    / "warehouse"
    / "duckdb"
    / "lendingclub.duckdb"
)

conn = duckdb.connect(str(DB_PATH))

print(DB_PATH)

D:\Project_Lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb


In [3]:
snapshot = conn.execute(
    """
    select
        count(*) as row_count
    from raw.lendingclub_raw
    """
).fetchdf()

snapshot

,row_count
0,2260701


In [4]:
columns_df = conn.execute(
    """
    select
        column_name,
        data_type
    from information_schema.columns
    where table_schema='raw'
      and table_name='lendingclub_raw'
    order by ordinal_position
    """
).fetchdf()

columns_df

,column_name,data_type
0,id,VARCHAR
1,member_id,VARCHAR
2,loan_amnt,DOUBLE
3,funded_amnt,DOUBLE
4,funded_amnt_inv,DOUBLE
5,term,VARCHAR
6,int_rate,DOUBLE
7,installment,DOUBLE
8,grade,VARCHAR
9,sub_grade,VARCHAR


In [5]:
len(columns_df)

151

In [6]:
conn.execute(
    """
    select *
    from raw.lendingclub_raw
    limit 5
    """
).fetchdf()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,None,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,None,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,w,0.00,0.00,4421.723917,4421.72,3600.00,821.72,0.0,0.0,0.0,Jan-2019,122.67,None,Mar-2019,564.0,560.0,0.0,30.0,1.0,Individual,NaN,NaN,None,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,Cash,N,None,None,None,NaN,NaN,NaN
1,68355089,None,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,None,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,w,0.00,0.00,25679.660000,25679.66,24700.00,979.66,0.0,0.0,0.0,Jun-2016,926.35,None,Mar-2019,699.0,695.0,0.0,NaN,1.0,Individual,NaN,NaN,None,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,NaN,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,Na

In [7]:
conn.execute(
    """
    select
        count(*) total_rows,
        count(distinct id) distinct_id
    from raw.lendingclub_raw
    """
).fetchdf()

,total_rows,distinct_id
0,2260701,2260701


In [8]:
conn.execute(
    """
    select
        count(*) total_rows,
        count(member_id) populated_member_id
    from raw.lendingclub_raw
    """
).fetchdf()

,total_rows,populated_member_id
0,2260701,0


In [9]:
candidate_columns = [
    "id",
    "member_id",
    "url",
    "loan_amnt"
]

for col in candidate_columns:

    try:

        df = conn.execute(
            f"""
            select
                count(*) total_rows,
                count(distinct {col}) distinct_values
            from raw.lendingclub_raw
            """
        ).fetchdf()

        print("\n")
        print(col)

        display(df)

    except Exception as e:

        print(col, e)



id


,total_rows,distinct_values
0,2260701,2260701




member_id


,total_rows,distinct_values
0,2260701,0




url


,total_rows,distinct_values
0,2260701,2260668




loan_amnt


,total_rows,distinct_values
0,2260701,1572


In [10]:
for c in columns_df["column_name"]:
    print(c)

id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
to

In [11]:
nulls_df = conn.execute(
    """
    select *
    from raw.lendingclub_raw
    
    """
).fetchdf()

(
    nulls_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
    
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

member_id                                     2260701
orig_projected_additional_accrued_interest    2252050
hardship_reason                               2249784
hardship_payoff_balance_amount                2249784
hardship_last_payment_amount                  2249784
payment_plan_start_date                       2249784
hardship_type                                 2249784
hardship_status                               2249784
hardship_start_date                           2249784
deferral_term                                 2249784
hardship_amount                               2249784
hardship_dpd                                  2249784
hardship_loan_status                          2249784
hardship_length                               2249784
hardship_end_date                             2249784
settlement_status                             2226455
debt_settlement_flag_date                     2226455
settlement_term                               2226455
settlement_percentage       

In [12]:
conn.execute(
    """
    select
        loan_status,
        count(*) loans
    from raw.lendingclub_raw
    group by 1
    order by 2 desc
    """
).fetchdf()

,loan_status,loans
0,Fully Paid,1076751
1,Current,878317
2,Charged Off,268559
3,Late (31-120 days),21467
4,In Grace Period,8436
5,Late (16-30 days),4349
6,Does not meet the credit policy. Status:Fully ...,1988
7,Does not meet the credit policy. Status:Charge...,761
8,Default,40
9,None,33


In [13]:
conn.execute(
    """
    select *
    from raw.lendingclub_raw
    where id is null
    """
).fetchdf()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term


In [14]:
conn.execute(
    """
    select
        min(issue_d),
        max(issue_d)
    from raw.lendingclub_raw
    """
).fetchdf()

,min(issue_d),max(issue_d)
0,Apr-2008,Sep-2018


In [15]:
conn.execute(
    """
    select
        application_type,
        count(*) loans
    from raw.lendingclub_raw
    group by 1
    order by 2 desc
    """
).fetchdf()

,application_type,loans
0,Individual,2139958
1,Joint App,120710
2,None,33


In [16]:
conn.execute(
    """
    select
        id,
        loan_status,
        issue_d
    from raw.lendingclub_raw
    where try_cast(id as bigint) is null
    """
).fetchdf()

,id,loan_status,issue_d
0,Total amount funded in policy code 1: 6417608175,None,None
1,Total amount funded in policy code 2: 1944088810,None,None
2,Total amount funded in policy code 1: 1741781700,None,None
3,Total amount funded in policy code 2: 564202131,None,None
4,Total amount funded in policy code 1: 1791201400,None,None
5,Total amount funded in policy code 2: 651669342,None,None
6,Total amount funded in policy code 1: 1443412975,None,None
7,Total amount funded in policy code 2: 511988838,None,None
8,Total amount funded in policy code 1: 2063142975,None,None
9,Total amount funded in policy code 2: 823319310,None,None


In [17]:
conn.execute(
    """
    select
        count(*)
    from raw.lendingclub_raw
    where try_cast(id as bigint) is null
    """
).fetchdf()

,count_star()
0,33


In [18]:
conn.execute(
    """
    select
        typeof(id)
    from raw.lendingclub_raw
    limit 1
    """
).fetchdf()

,typeof(id)
0,VARCHAR


In [19]:
conn.execute(
    """
    select
        count(*) as loans,
        count(distinct id) as distinct_id
    from raw.lendingclub_raw
    where try_cast(id as bigint) is not null
    """
).fetchdf()

,loans,distinct_id
0,2260668,2260668


In [20]:
conn.execute(
    """
    select
        min(issue_d) as min_issue_d,
        max(issue_d) as max_issue_d
    from raw.lendingclub_raw
    where try_cast(id as bigint) is not null
    """
).fetchdf()

,min_issue_d,max_issue_d
0,Apr-2008,Sep-2018


In [21]:
conn.execute(
    """
    select
        application_type,
        count(*) loans
    from raw.lendingclub_raw
    where try_cast(id as bigint) is not null
    group by 1
    order by 2 desc
    """
).fetchdf()

,application_type,loans
0,Individual,2139958
1,Joint App,120710


In [22]:
loan_status_df = conn.execute(
    """
    select
        loan_status,
        count(*) loans
    from raw.lendingclub_raw
    where try_cast(id as bigint) is not null
    group by 1
    order by 2 desc
    """
).fetchdf()

loan_status_df["pct"] = (
    loan_status_df["loans"]
    / loan_status_df["loans"].sum()
    * 100
).round(2)

loan_status_df

,loan_status,loans,pct
0,Fully Paid,1076751,47.63
1,Current,878317,38.85
2,Charged Off,268559,11.88
3,Late (31-120 days),21467,0.95
4,In Grace Period,8436,0.37
5,Late (16-30 days),4349,0.19
6,Does not meet the credit policy. Status:Fully ...,1988,0.09
7,Does not meet the credit policy. Status:Charge...,761,0.03
8,Default,40,0.00


In [23]:
conn.execute(
    """
    select
        policy_code,
        count(*) loans
    from raw.lendingclub_raw
    where try_cast(id as bigint) is not null
    group by 1
    order by 1
    """
).fetchdf()

,policy_code,loans
0,1.0,2260668


In [24]:
conn.execute(
    """
    select
        grade,
        count(*) loans
    from raw.lendingclub_raw
    where try_cast(id as bigint) is not null
    group by 1
    order by 1
    """
).fetchdf()

,grade,loans
0,A,433027
1,B,663557
2,C,650053
3,D,324424
4,E,135639
5,F,41800
6,G,12168


In [25]:
conn.execute(
    """
    select
        count(*) duplicate_loans
    from
    (
        select
            id,
            count(*) cnt
        from raw.lendingclub_raw
        where try_cast(id as bigint) is not null
        group by id
        having count(*) > 1
    )
    """
).fetchdf()

,duplicate_loans
0,0


In [26]:
profile_df = conn.execute(
    """
    select *
    from raw.lendingclub_raw
    where try_cast(id as bigint) is not null
    """
).fetchdf()

summary = []

for col in profile_df.columns:

    summary.append(
        {
            "column_name": col,
            "dtype": str(profile_df[col].dtype),
            "rows": len(profile_df),
            "nulls": profile_df[col].isna().sum(),
            "null_pct": round(
                profile_df[col].isna().mean() * 100,
                2
            ),
            "distinct_values":
                profile_df[col].nunique(dropna=True)
        }
    )

profile_summary = (
    pd.DataFrame(summary)
    .sort_values(
        ["null_pct", "column_name"],
        ascending=[False, True]
    )
)

profile_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,dtype,rows,nulls,null_pct,distinct_values
1,member_id,object,2260668,2260668,100.00,0
140,orig_projected_additional_accrued_interest,float64,2260668,2252017,99.62,7487
132,deferral_term,float64,2260668,2249751,99.52,1
133,hardship_amount,float64,2260668,2249751,99.52,9162
138,hardship_dpd,float64,2260668,2249751,99.52,34
135,hardship_end_date,object,2260668,2249751,99.52,28
142,hardship_last_payment_amount,float64,2260668,2249751,99.52,9045
137,hardship_length,float64,2260668,2249751,99.52,1
139,hardship_loan_status,object,2260668,2249751,99.52,5
141,hardship_payoff_balance_amount,float64,2260668,2249751,99.52,10893


# Clean Layer Specification v1

## Objective

Transform raw LendingClub data into a trusted clean dataset.

## Source

raw.lendingclub_raw

## Target

clean.lendingclub_clean

## Cleaning Rules

### Rule 1

Remove footer rows.

Condition:

```sql
try_cast(id as bigint) is not null
```

### Rule 2

Drop member_id.

Reason:

100% null.

### Rule 3

Drop policy_code.

Reason:

Single value.

### Rule 4

Standardize date columns.

Convert:

- issue_d
- earliest_cr_line
- last_pymnt_d
- next_pymnt_d
- last_credit_pull_d
- hardship_start_date
- hardship_end_date
- payment_plan_start_date
- settlement_date
- debt_settlement_flag_date

to proper date types.

### Rule 5

Preserve sparse columns.

Do not remove:

- hardship_*
- settlement_*
- joint applicant columns

Reason:

Business meaning is important despite sparsity.

### Rule 6

Preserve original values.

No imputation in clean layer.

Imputation belongs to feature engineering.

### Rule 7

Maintain loan-level grain.

1 row = 1 loan.

In [13]:
import duckdb

conn = duckdb.connect(
    r"D:\project_lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb"
)

In [6]:
conn.execute("""
select count(*)
from core.loan_master
""").fetchdf()

,count_star()
0,2260668


In [7]:
conn.execute("""
describe core.loan_master
""").fetchdf().head(10)

,column_name,column_type,null,key,default,extra
0,loan_id,BIGINT,YES,None,None,None
1,loan_amnt,DOUBLE,YES,None,None,None
2,funded_amnt,DOUBLE,YES,None,None,None
3,funded_amnt_inv,DOUBLE,YES,None,None,None
4,term,VARCHAR,YES,None,None,None
5,int_rate,DOUBLE,YES,None,None,None
6,installment,DOUBLE,YES,None,None,None
7,grade,VARCHAR,YES,None,None,None
8,sub_grade,VARCHAR,YES,None,None,None
9,emp_title,VARCHAR,YES,None,None,None


In [9]:
conn.execute("""
select
    issue_d,
    count(*) as loans
from core.loan_master
group by 1
order by 1

""").fetchdf()

,issue_d,loans
0,Apr-2008,259
1,Apr-2009,333
2,Apr-2010,912
3,Apr-2011,1563
4,Apr-2012,3230
5,Apr-2013,9419
6,Apr-2014,19071
7,Apr-2015,35427
8,Apr-2016,36432
9,Apr-2017,29683


In [10]:
date_cols = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

for c in date_cols:
    print("\n", c)
    print(
        conn.execute(
            f"""
            select distinct {c}
            from core.loan_master
            where {c} is not null
            limit 10
            """
        ).fetchdf()
    )


 issue_d
    issue_d
0  Jul-2015
1  Jun-2017
2  Apr-2017
3  Feb-2010
4  Feb-2009
5  Dec-2008
6  Apr-2008
7  Oct-2010
8  Sep-2010
9  Apr-2013

 earliest_cr_line
  earliest_cr_line
0         Oct-1987
1         Sep-2001
2         Apr-2001
3         Nov-2001
4         Jan-2001
5         Dec-1987
6         Jun-2004
7         Dec-2002
8         Jan-2002
9         Jul-1995

 last_pymnt_d
  last_pymnt_d
0     Jun-2016
1     Aug-2018
2     Nov-2018
3     Aug-2016
4     Nov-2017
5     Jul-2018
6     Feb-2016
7     Apr-2014
8     Oct-2014
9     Apr-2011

 next_pymnt_d
  next_pymnt_d
0     Nov-2013
1     Sep-2012
2     May-2012
3     Dec-2015
4     May-2011
5     Dec-2009
6     Jun-2008
7     Feb-2008
8     Sep-2013
9     May-2014

 last_credit_pull_d
  last_credit_pull_d
0           Oct-2017
1           Mar-2017
2           Oct-2016
3           May-2017
4           May-2016
5           Dec-2015
6           Nov-2013
7           Sep-2012
8           May-2012
9           May-2011


In [11]:
date_cols = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

for col in date_cols:

    result = conn.execute(
        f"""
        select
            count(*) total_rows,
            count({col}) populated_rows
        from core.loan_master
        """
    ).fetchdf()

    print("\n", col)
    print(result)


 issue_d
   total_rows  populated_rows
0     2260668         2260668

 earliest_cr_line
   total_rows  populated_rows
0     2260668         2260639

 last_pymnt_d
   total_rows  populated_rows
0     2260668         2258241

 next_pymnt_d
   total_rows  populated_rows
0     2260668          915358

 last_credit_pull_d
   total_rows  populated_rows
0     2260668         2260596


In [ ]:
conn.close()

print("Connection closed")

Connection closed


: 

In [14]:
conn.execute("""
describe core.loan_master_v2
""").fetchdf().head(15)

,column_name,column_type,null,key,default,extra
0,loan_id,BIGINT,YES,None,None,None
1,issue_date,TIMESTAMP,YES,None,None,None
2,earliest_credit_date,TIMESTAMP,YES,None,None,None
3,last_payment_date,TIMESTAMP,YES,None,None,None
4,next_payment_date,TIMESTAMP,YES,None,None,None
5,last_credit_pull_date,TIMESTAMP,YES,None,None,None
6,loan_amnt,DOUBLE,YES,None,None,None
7,funded_amnt,DOUBLE,YES,None,None,None
8,funded_amnt_inv,DOUBLE,YES,None,None,None
9,term,VARCHAR,YES,None,None,None
